# **Download Libraries & Dataset**

In [ ]:
!pip install open_clip_torch
!pip install qwen-vl-utils

In [ ]:
!pip install kaggle

In [ ]:
!kaggle auth login

In [ ]:
!kaggle kernels pull ipythonx/mvtec-ad-anomaly-detection-with-anomalib-library

In [ ]:
!pip install kaggle

!kaggle datasets download -d ipythonx/mvtec-ad
!unzip mvtec-ad.zip -d /content/mvtec_ad/

# **CLIP - All Classes**

In [ ]:
"""
1. Loads a pretrained CLIP model (no training, no fine-tuning -> true zero-shot).
2. For each object category, scores every test image as normal/anomalous using
   an ENSEMBLE of text prompts (WinCLIP-style zero-shot AD recipe).
3. Computes image-level AUROC per category (standard MVTec-AD metric).
4. Computes a simple "trustworthiness" signal: the variance across the prompt
   ensemble's individual scores, used as a per-prediction uncertainty estimate.
   We then check whether high uncertainty correlates with wrong predictions.
"""

import argparse
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

import open_clip


# ---------------------------------------------------------------------------
# Prompt ensemble (WinCLIP-style). 
# Multiple phrasings per state ->  average the embeddings for the final score, and ALSO keep per-template scores to
# compute the uncertainty signal.
# ---------------------------------------------------------------------------
NORMAL_TEMPLATES = [
    "a photo of a normal {}.",
    "a photo of a flawless {}.",
    "a photo of a {} without any defect.",
    "a cropped photo of a normal {}.",
    "an industrial photo of an undamaged {}.",
]

ANOMALY_TEMPLATES = [
    "a photo of a damaged {}.",
    "a photo of a {} with a defect.",
    "a photo of a flawed {}.",
    "a cropped photo of an anomalous {}.",
    "an industrial photo of a broken {}.",
]


def build_text_embeddings(model, tokenizer, class_name, device):
    normal_prompts = [t.format(class_name) for t in NORMAL_TEMPLATES]
    anomaly_prompts = [t.format(class_name) for t in ANOMALY_TEMPLATES]

    with torch.no_grad():
        normal_tokens = tokenizer(normal_prompts).to(device)
        anomaly_tokens = tokenizer(anomaly_prompts).to(device)
        normal_embeds = model.encode_text(normal_tokens)
        anomaly_embeds = model.encode_text(anomaly_tokens)
        normal_embeds = normal_embeds / normal_embeds.norm(dim=-1, keepdim=True)
        anomaly_embeds = anomaly_embeds / anomaly_embeds.norm(dim=-1, keepdim=True)

    return normal_embeds, anomaly_embeds  # [n_templates, dim] each


def score_image(model, preprocess, image_path, normal_embeds, anomaly_embeds, device):
    """Returns (mean_anomaly_score, uncertainty) for one image.
    mean_anomaly_score: average over templates of (sim_anomaly - sim_normal)
    uncertainty: std dev of the per-template anomaly score (prompt-ensemble disagreement)
    """
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        img_embed = model.encode_image(image)
        img_embed = img_embed / img_embed.norm(dim=-1, keepdim=True)

        sim_normal = (img_embed @ normal_embeds.T).squeeze(0)   # [n_templates]
        sim_anomaly = (img_embed @ anomaly_embeds.T).squeeze(0)  # [n_templates]

        per_template_score = (sim_anomaly - sim_normal).cpu().numpy()

    return float(per_template_score.mean()), float(per_template_score.std())


def run_category(model, preprocess, tokenizer, data_root, category, device):
    class_name = category.replace("_", " ")
    normal_embeds, anomaly_embeds = build_text_embeddings(model, tokenizer, class_name, device)

    test_dir = Path(data_root) / category / "test"
    records = []

    for subfolder in sorted(test_dir.iterdir()):
        if not subfolder.is_dir():
            continue
        label = 0 if subfolder.name == "good" else 1  # 0 = normal, 1 = anomalous
        image_paths = sorted(list(subfolder.glob("*.png")) + list(subfolder.glob("*.jpg")))

        for img_path in tqdm(image_paths, desc=f"{category}/{subfolder.name}", leave=False):
            score, uncertainty = score_image(
                model, preprocess, img_path, normal_embeds, anomaly_embeds, device
            )
            records.append(
                {
                    "category": category,
                    "defect_type": subfolder.name,
                    "path": str(img_path),
                    "label": label,
                    "anomaly_score": score,
                    "uncertainty": uncertainty,
                }
            )

    return pd.DataFrame(records)


def summarize(df):
    results = []
    for category, group in df.groupby("category"):
        auroc = roc_auc_score(group["label"], group["anomaly_score"])

        threshold = group["anomaly_score"].median()
        pred = (group["anomaly_score"] > threshold).astype(int)
        correct = (pred == group["label"]).astype(int)

        unc_correct = group.loc[correct == 1, "uncertainty"].mean()
        unc_wrong = group.loc[correct == 0, "uncertainty"].mean()

        results.append(
            {
                "category": category,
                "n_images": len(group),
                "AUROC": round(auroc, 4),
                "mean_uncertainty_correct_preds": round(unc_correct, 4),
                "mean_uncertainty_wrong_preds": round(unc_wrong, 4)
                if not np.isnan(unc_wrong)
                else None,
            }
        )
    return pd.DataFrame(results)


def main(argv=None):
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--data_root",
        type=str,
        default="/content/mvtec_ad", 
    )

    parser.add_argument(
        "--categories",
        type=str,
        nargs="+",
        default=["bottle", "hazelnut", "grid", "toothbrush", "cable", "capsule",
                "carpet", "leather", "metal_nut", "pill", "screw", "tile", "wood",
                "transistor", "zipper"],
    )

    parser.add_argument(
        "--model_name",
        type=str,
        default="ViT-B-16",
    )

    parser.add_argument(
        "--pretrained",
        type=str,
        default="openai",
    )

    parser.add_argument(
        "--output_dir",
        type=str,
        default="./results",
    )

    if argv is None:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args(argv)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    model, _, preprocess = open_clip.create_model_and_transforms(
        args.model_name,
        pretrained=args.pretrained,
    )
    tokenizer = open_clip.get_tokenizer(args.model_name)
    model = model.to(device).eval()

    os.makedirs(args.output_dir, exist_ok=True)

    all_dfs = []

    for category in args.categories:
        print(f"\n=== Processing category: {category} ===")
        df = run_category(
            model,
            preprocess,
            tokenizer,
            args.data_root,
            category,
            device,
        )
        all_dfs.append(df)

    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df.to_csv(
        os.path.join(args.output_dir, "raw_scores.csv"),
        index=False,
    )

    summary_df = summarize(full_df)
    summary_df.to_csv(
        os.path.join(args.output_dir, "summary_auroc.csv"),
        index=False,
    )

    print("\n=== SUMMARY ===")
    print(summary_df.to_string(index=False))
    print(f"\nOverall mean AUROC: {summary_df['AUROC'].mean():.4f}")

if __name__ == "__main__":
    main()

    # main([
    # "--data_root", "/content/mvtec_ad",
    # "--categories", "bottle", "hazelnut", "capsule",
    # ])

In [ ]:
full_df = pd.read_csv("/content/results/raw_scores.csv")
full_df

### Visualize

In [ ]:
import os
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


# ------------------------------------------------------------
# Create prediction and TP/TN/FP/FN labels
# ------------------------------------------------------------
def prepare_predictions(df):

    df = df.copy()
    df["prediction"] = 0

    for category in df["category"].unique():
        mask = df["category"] == category

        threshold = df.loc[mask, "anomaly_score"].median()

        df.loc[mask, "prediction"] = (
            df.loc[mask, "anomaly_score"] > threshold
        ).astype(int)

    df["prediction_type"] = np.select(
        [
            (df["label"] == 1) & (df["prediction"] == 1),
            (df["label"] == 0) & (df["prediction"] == 0),
            (df["label"] == 1) & (df["prediction"] == 0),
            (df["label"] == 0) & (df["prediction"] == 1),
        ],
        [
            "TP",
            "TN",
            "FN",
            "FP",
        ],
        default="Unknown",
    )

    return df


# ------------------------------------------------------------
# Ground-truth mask
# ------------------------------------------------------------
def get_mask_path(img_path):
    """
    Returns the corresponding MVTec ground-truth mask.
    """

    p = Path(img_path)

    if p.parent.name == "good":
        return None

    mask_path = Path(
        str(p).replace(
            f"{os.sep}test{os.sep}",
            f"{os.sep}ground_truth{os.sep}",
        )
    )

    mask_path = mask_path.with_name(
        mask_path.stem + "_mask" + mask_path.suffix
    )

    return mask_path if mask_path.exists() else None


# ------------------------------------------------------------
# text wrapping
# ------------------------------------------------------------
def wrap_caption(text, width=46, max_lines=8):

    lines = textwrap.wrap(str(text), width=width)

    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] += "..."

    return "\n".join(lines)


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------
def display_predictions(
    df,
    n=6,
    category=None,
    defect_type=None,
    filter_query=None,
    sort_by=None,
    ascending=True,
    row_height=3.2,
    save_path=None,
):

    data = df.copy()

    if category is not None:
        data = data[data["category"] == category]

    if defect_type is not None:
        data = data[data["defect_type"] == defect_type]

    if filter_query is not None:
        data = data.query(filter_query)

    if sort_by is not None:
        data = data.sort_values(sort_by, ascending=ascending)

    data = data.head(n)

    if len(data) == 0:
        print("No rows matched.")
        return

    fig, axes = plt.subplots(
        len(data),
        4,
        figsize=(14, row_height * len(data)),
    )

    if len(data) == 1:
        axes = axes.reshape(1, -1)

    for i, (_, row) in enumerate(data.iterrows()):

        img = Image.open(row["path"]).convert("RGB")
        img_arr = np.array(img)

        mask_path = get_mask_path(row["path"])

        if mask_path is not None:
            mask_arr = np.array(Image.open(mask_path).convert("L"))
        else:
            mask_arr = None

        # ----------------------------------------------------
        # Original image
        # ----------------------------------------------------
        axes[i, 0].imshow(img_arr)
        axes[i, 0].set_title(
            f"{row['category']}/{row['defect_type']}\nGT={row['label']}",
            fontsize=9,
        )
        axes[i, 0].axis("off")

        # ----------------------------------------------------
        # Ground-truth mask
        # ----------------------------------------------------
        if mask_arr is not None:

            axes[i, 1].imshow(
                mask_arr,
                cmap="gray",
                vmin=0,
                vmax=255,
            )

            axes[i, 1].set_title("GT mask", fontsize=9)

        else:

            axes[i, 1].imshow(
                np.zeros(img_arr.shape[:2]),
                cmap="gray",
            )

            axes[i, 1].set_title(
                "GT mask (normal)",
                fontsize=9,
            )

        axes[i, 1].axis("off")

        # ----------------------------------------------------
        # Overlay
        # ----------------------------------------------------
        overlay = img_arr.copy()

        if mask_arr is not None:

            resized = np.array(
                Image.fromarray(mask_arr).resize(
                    (img_arr.shape[1], img_arr.shape[0]),
                    Image.NEAREST,
                )
            )

            tint = overlay.copy()
            tint[resized > 127] = [255, 0, 0]

            overlay = (
                0.6 * overlay +
                0.4 * tint
            ).astype(np.uint8)

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay", fontsize=9)
        axes[i, 2].axis("off")

        # ----------------------------------------------------
        # Prediction information
        # ----------------------------------------------------
        axes[i, 3].axis("off")

        pred_label = (
            "Anomaly"
            if row["prediction"] == 1
            else "Normal"
        )

        gt_label = (
            "Anomaly"
            if row["label"] == 1
            else "Normal"
        )

        info = (
            f"Prediction Type : {row['prediction_type']}\n\n"
            f"Ground Truth    : {gt_label}\n"
            f"Prediction      : {pred_label}\n\n"
            f"Anomaly Score   : {row['anomaly_score']:.4f}\n"
            f"Uncertainty     : {row['uncertainty']:.4f}"
        )

        axes[i, 3].text(
            0,
            1,
            info,
            fontsize=9,
            family="monospace",
            va="top",
        )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=150,
            bbox_inches="tight",
        )
        print(f"Saved to {save_path}")

    plt.show()

In [ ]:
full_df = prepare_predictions(full_df)

#### Carpet

In [ ]:
display_predictions(
    full_df,
    category="carpet",
    filter_query="prediction_type == 'TP'"
)

In [ ]:
display_predictions(
    full_df,
    category="carpet",
    filter_query="prediction_type == 'TN'"
)

In [ ]:
display_predictions(
    full_df,
    category="carpet",
    filter_query="prediction_type == 'FN'"
)

#### Pill

In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'TP'"
)

In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'FP'"
)

In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'TN'"
)

In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'FN'"
)

---
# **Reasoning VLMs**


## - ***Qwen2.5-VL-7B-Instruct***

In [ ]:
!pip install qwen-vl-utils

In [ ]:
  # !pip install -q transformers accelerate qwen-vl-utils pillow scikit-learn pandas tqdm

### > All Classes

In [ ]:
full_df["prediction_type"] = np.select(
    [
        (full_df["label"] == 1) & (full_df["anomaly_score"] == 1),
        (full_df["label"] == 0) & (full_df["anomaly_score"] == 0),
        (full_df["label"] == 1) & (full_df["anomaly_score"] == 0),
        (full_df["label"] == 0) & (full_df["anomaly_score"] == 1),
    ],
    [
        "TP",
        "TN",
        "FN",
        "FP",
    ],
    default="Unknown",
)

In [ ]:
full_df["prediction_type"].value_counts()

In [ ]:
import math
import os
import random
import re
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info


# ============================================================
# CONFIG
# ============================================================
CONFIG = SimpleNamespace(
    data_root="/content/mvtec_ad",
    categories=["bottle", "hazelnut", "grid", "toothbrush", "cable", "capsule",
                "carpet", "leather", "metal_nut", "pill", "screw", "tile", "wood",
                "transistor", "zipper"],
    model_id="Qwen/Qwen2.5-VL-7B-Instruct",
    output_dir="/content/results3_vlm",
    max_images_per_class=20,   # subsample per defect-type folder
    n_samples=10,               # more votes = more stable fraction
    max_new_tokens=150,
    temperature=0.6,           # need real sampling diversity for votes to disagree meaningfully
)


ANSWER_RE = re.compile(r"ANSWER:\s*(YES|NO)", re.IGNORECASE)
OBSERVATION_RE = re.compile(r"OBSERVATION:\s*(.+?)(?=DEFECT_TYPE:|ANSWER:|$)", re.IGNORECASE | re.DOTALL)
DEFECT_TYPE_RE = re.compile(r"DEFECT_TYPE:\s*(.+?)(?=DEFECT_LOCATION:|ANSWER:|$)", re.I | re.S)
LOCATION_RE = re.compile(r"DEFECT_LOCATION:\s*(.+?)(?=CONFIDENCE:|ANSWER:|$)", re.I | re.S)
CONFIDENCE_RE = re.compile(r"CONFIDENCE:\s*(HIGH|MEDIUM|LOW)", re.I)

# Canonicalization map for defect-type strings, used only for the semantic-entropy signal 
# so that trivial wording differences ("broken" vs "broken part") don't get counted as disagreement.
DEFECT_CANON = {
    "": "none", "none": "none", "n/a": "none", "no defect": "none",
    "broken": "broken part", "break": "broken part", "broken part": "broken part",
    "crack": "crack", "cracked": "crack",
    "scratch": "scratch", "scratched": "scratch",
    "hole": "hole",
    "dent": "dent", "dented": "dent",
    "contamination": "contamination", "contaminated": "contamination",
    "deformation": "deformation", "deformed": "deformation",
    "discoloration": "discoloration", "discolouration": "discoloration",
}
CONFIDENCE_MAP = {"HIGH": 1.0, "MEDIUM": 0.5, "LOW": 0.0}


def build_prompt(class_name):
    return f"""
            You are an expert industrial quality inspector evaluating a manufactured {class_name} from the MVTec AD benchmark.

            Inspect the object carefully.

            Only report defects that are clearly visible.

            Do NOT guess.

            If you cannot confidently determine whether the object is normal or defective, report the uncertainty explicitly and avoid inventing defects.

            Possible defects include:
            - crack
            - scratch
            - hole
            - dent
            - contamination
            - broken part
            - deformation
            - discoloration

            Respond EXACTLY in this format:

            OBSERVATION:
            <what you visually observe>

            DEFECT_TYPE:
            <one word or NONE>

            DEFECT_LOCATION:
            <location or NONE>

            CONFIDENCE:
            HIGH
            MEDIUM
            LOW

            ANSWER:
            YES
            or
            NO
            """.strip()


def load_model(model_id, device):
    print(f"Loading {model_id} ... (first run downloads weights, can take a few minutes)")

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map="auto"
    )

    processor = AutoProcessor.from_pretrained(model_id)
    model.eval()
    return model, processor


def query_vlm(model, processor, image_path, class_name, n_samples, max_new_tokens, temperature):
    """Returns list of dicts: vote (1/0/None), observation, defect_type, location, confidence."""
    image = Image.open(image_path).convert("RGB")
    prompt_text = build_prompt(class_name)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[chat_text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            num_return_sequences=n_samples,
        )

    prompt_len = inputs.input_ids.shape[1]
    trimmed = generated_ids[:, prompt_len:]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    results = []
    for text in decoded:
        answer = ANSWER_RE.search(text)
        observation = OBSERVATION_RE.search(text)
        defect = DEFECT_TYPE_RE.search(text)
        location = LOCATION_RE.search(text)
        confidence = CONFIDENCE_RE.search(text)

        vote = None
        if answer:
            vote = 1 if answer.group(1).upper() == "YES" else 0

        results.append({
            "vote": vote,
            "observation": observation.group(1).strip() if observation else "",
            "defect_type": defect.group(1).strip() if defect else "",
            "location": location.group(1).strip() if location else "",
            "confidence": confidence.group(1).upper() if confidence else "",
        })

    return results


def normalize_defect(defect_str):
    d = (defect_str or "").strip().lower()
    return DEFECT_CANON.get(d, d)


def semantic_self_consistency(samples):
    """
    Computes two uncertainty signals that use the *content* of repeated
    samples, not just the YES/NO fraction:

    - semantic_entropy: normalized entropy over (vote, canonical defect_type)
      pairs across the n samples. High when the model tells a different
      story each time (different verdict and/or different defect named),
      low when it repeats the same verdict+defect consistently
      -- even if that verdict happens to be wrong.

    - confidence_uncertainty: 1 - mean(self-reported confidence), using the
      model's own HIGH/MEDIUM/LOW confidence tag, mapped to 1.0/0.5/0.0.

    Returns (semantic_entropy, confidence_uncertainty), each None if there
    is no valid data to compute it from.
    """
    signatures = []
    conf_values = []

    for s in samples:
        if s["vote"] is None:
            continue
        signatures.append((s["vote"], normalize_defect(s["defect_type"])))
        if s["confidence"] in CONFIDENCE_MAP:
            conf_values.append(CONFIDENCE_MAP[s["confidence"]])

    n = len(signatures)
    if n == 0:
        return None, None

    counts = Counter(signatures)
    entropy = -sum((c / n) * math.log(c / n) for c in counts.values())
    max_entropy = math.log(n) if n > 1 else 1.0
    semantic_entropy = entropy / max_entropy if max_entropy > 0 else 0.0

    confidence_uncertainty = (1 - (sum(conf_values) / len(conf_values))) if conf_values else None

    return semantic_entropy, confidence_uncertainty


def run_category(model, processor, data_root, category, cfg):
    class_name = category.replace("_", " ")
    test_dir = Path(data_root) / category / "test"
    records = []
    rng = random.Random(42)

    for subfolder in sorted(test_dir.iterdir()):
        if not subfolder.is_dir():
            continue
        label = 0 if subfolder.name == "good" else 1
        image_paths = sorted(list(subfolder.glob("*.png")) + list(subfolder.glob("*.jpg")))

        if cfg.max_images_per_class and len(image_paths) > cfg.max_images_per_class:
            image_paths = rng.sample(image_paths, cfg.max_images_per_class)

        for img_path in tqdm(image_paths, desc=f"{category}/{subfolder.name}", leave=False):
            samples = query_vlm(
                model, processor, img_path, class_name,
                n_samples=cfg.n_samples, max_new_tokens=cfg.max_new_tokens,
                temperature=cfg.temperature,
            )
            votes = [s["vote"] for s in samples if s["vote"] is not None]
            observations = [s["observation"] for s in samples]
            defect_types = [s["defect_type"] for s in samples]
            locations = [s["location"] for s in samples]
            confidences = [s["confidence"] for s in samples]

            if not votes:
                continue 

            anomaly_score = sum(votes) / len(votes)          # fraction of YES votes
            p = anomaly_score
            verdict_variance = (p * (1 - p)) ** 0.5          

            semantic_entropy, confidence_uncertainty = semantic_self_consistency(samples)

            records.append({
                "category": category,
                "defect_type": subfolder.name,
                "path": str(img_path),
                "label": label,
                "anomaly_score": anomaly_score,
                "verdict_variance": verdict_variance,
                "semantic_entropy": semantic_entropy,
                "confidence_uncertainty": confidence_uncertainty,
                "n_valid_votes": len(votes),

                "example_observation": observations[0],
                "predicted_defect": defect_types[0],
                "predicted_location": locations[0],
                "predicted_confidence": confidences[0],
            })

    return pd.DataFrame(records)


def summarize(df):
    unc_cols = ["verdict_variance", "semantic_entropy", "confidence_uncertainty"]
    results = []

    for category, group in df.groupby("category"):
        if group["label"].nunique() < 2:
            print(f"Skipping AUROC for {category}: only one class present in sample.")
            continue

        auroc = roc_auc_score(group["label"], group["anomaly_score"])
        pred = (group["anomaly_score"] > 0.5).astype(int)
        correct = (pred == group["label"]).astype(int)

        row = {"category": category, "n_images": len(group), "AUROC": round(auroc, 4)}

        for col in unc_cols:
            valid = group[col].notna()
            unc_correct = group.loc[valid & (correct == 1), col].mean()
            unc_wrong = group.loc[valid & (correct == 0), col].mean()
            row[f"{col}_correct"] = round(unc_correct, 4) if pd.notna(unc_correct) else None
            row[f"{col}_wrong"] = round(unc_wrong, 4) if pd.notna(unc_wrong) else None

        results.append(row)

    return pd.DataFrame(results)


def main(cfg):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    if device == "cpu":
        print("WARNING: no GPU detected. This will be very slow. Use a Colab GPU runtime.")

    model, processor = load_model(cfg.model_id, device)
    os.makedirs(cfg.output_dir, exist_ok=True)

    all_dfs = []
    for category in cfg.categories:
        print(f"\n=== Processing category: {category} ===")
        df = run_category(model, processor, cfg.data_root, category, cfg)
        all_dfs.append(df)

    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df.to_csv(os.path.join(cfg.output_dir, "raw_scores_vlm.csv"), index=False)

    summary_df = summarize(full_df)
    summary_df.to_csv(os.path.join(cfg.output_dir, "summary_auroc_vlm.csv"), index=False)

    print("\n=== SUMMARY (also saved to results_vlm/summary_auroc_vlm.csv) ===")
    print(summary_df.to_string(index=False))
    if len(summary_df):
        print(f"\nOverall mean AUROC: {summary_df['AUROC'].mean():.4f}")

    print("\n=== Example observation ===")
    anomalous_examples = full_df[full_df["label"] == 1].head(2)
    normal_examples = full_df[full_df["label"] == 0].head(2)
    for _, row in pd.concat([anomalous_examples, normal_examples]).iterrows():
        print(
            f"[{row['category']}/{row['defect_type']}] score={row['anomaly_score']:.2f} "
            f"verdict_var={row['verdict_variance']:.2f} "
            f"sem_ent={row['semantic_entropy']:.2f} "
            f"-> {row['example_observation']}"
        )
        print(
            f"""
                [{row['category']}/{row['defect_type']}]

                GT label: {row['label']}
                Score: {row['anomaly_score']:.2f}
                Confidence: {row['predicted_confidence']}

                Observation:
                {row['example_observation']}

                Predicted defect:
                {row['predicted_defect']}

                Location:
                {row['predicted_location']}
                """
        )

    return full_df, summary_df


full_df, summary_df = main(CONFIG)

In [ ]:
full_df = pd.read_csv("/content/results3_vlm/raw_scores_vlm.csv")
full_df

In [ ]:
full_df["prediction_type"] = np.select(
    [
        (full_df["label"] == 1) & (full_df["anomaly_score"] == 1),
        (full_df["label"] == 0) & (full_df["anomaly_score"] == 0),
        (full_df["label"] == 1) & (full_df["anomaly_score"] == 0),
        (full_df["label"] == 0) & (full_df["anomaly_score"] == 1),
    ],
    [
        "TP",
        "TN",
        "FN",
        "FP",
    ],
    default="Unknown",
)

In [ ]:
full_df["prediction_type"].value_counts()

In [ ]:
fp_df = full_df[full_df["prediction_type"] == "FP"]
fp_df.head(5)

In [ ]:
category_summary = (
    full_df.groupby(["category", "prediction_type"])
           .size()
           .unstack(fill_value=0)
)

category_summary["Correct"] = (
    category_summary.get("TP", 0)
    + category_summary.get("TN", 0)
)

category_summary["Wrong"] = (
    category_summary.get("FP", 0)
    + category_summary.get("FN", 0)
)

category_summary["Accuracy (%)"] = (
    100
    * category_summary["Correct"]
    / (category_summary["Correct"] + category_summary["Wrong"])
).round(2)

print(category_summary)

In [ ]:
import matplotlib.pyplot as plt

plot_df = category_summary[["TP", "TN", "FP", "FN"]]

plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color={
        "TP": "green",
        "TN": "blue",
        "FP": "orange",
        "FN": "red"
    }
)

plt.ylabel("Number of images")
plt.xlabel("Category")
plt.title("Prediction outcomes by category")
plt.xticks(rotation=45)
plt.legend(title="Prediction Type")
plt.tight_layout()
plt.show()

In [ ]:
errors = full_df[full_df["prediction_type"].isin(["FP", "FN"])]

error_summary = (
    errors.groupby(["category", "defect_type", "prediction_type"])
          .size()
          .unstack(fill_value=0)
          .sort_index()
)

print(error_summary)

#### Visualize

In [ ]:
import os
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def get_mask_path(img_path):
    p = Path(img_path)
    if p.parent.name == "good":
        return None
    mask_path = Path(str(p).replace(f"{os.sep}test{os.sep}", f"{os.sep}ground_truth{os.sep}"))
    mask_path = mask_path.with_name(mask_path.stem + "_mask" + mask_path.suffix)
    return mask_path if mask_path.exists() else None


def wrap_caption(text, width=46, max_lines=6):
    lines = textwrap.wrap(text or "", width=width)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip() + "..."
    return "\n".join(lines)


def display_predictions(
    df,
    n=6,
    category=None,
    defect_type=None,
    filter_query=None,
    sort_by=None,
    ascending=True,
    row_height=3.2,
    save_path=None,
):
    """
    Displays up to n rows from df as four panels per row:
    original image | ground-truth mask | overlay | prediction + caption text.

    Parameters
    ----------
    category : str, optional
        Filter by category (e.g. "bottle", "capsule").

    defect_type : str, optional
        Filter by defect type (e.g. "broken_small").

    filter_query : str, optional
        Additional pandas query string.

    sort_by : str, optional
        Column name to sort by.

    save_path : str, optional
        If given, saves the figure.
    """

    data = df.copy()

    # Filter by category
    if category is not None:
        data = data[data["category"] == category]

    # Filter by defect type
    if defect_type is not None:
        data = data[data["defect_type"] == defect_type]

    # custom query
    if filter_query:
        data = data.query(filter_query)

    # Sort if requested
    if sort_by:
        data = data.sort_values(sort_by, ascending=ascending)

    # Keep only first n samples
    data = data.head(n)

    if len(data) == 0:
        print("No rows matched.")
        return

    fig, axes = plt.subplots(
        len(data),
        4,
        figsize=(4 * 3.4, row_height * len(data))
    )

    if len(data) == 1:
        axes = axes.reshape(1, -1)

    for i, (_, row) in enumerate(data.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        img_arr = np.array(img)

        mask_path = get_mask_path(row["path"])
        mask_arr = (
            np.array(Image.open(mask_path).convert("L"))
            if mask_path else None
        )

        # Column 0: Original image
        axes[i, 0].imshow(img_arr)
        axes[i, 0].set_title(
            f"{row['category']}/{row['defect_type']}\nGT label: {row['label']}",
            fontsize=9,
        )
        axes[i, 0].axis("off")

        # Column 1: Ground-truth mask
        if mask_arr is not None:
            axes[i, 1].imshow(mask_arr, cmap="gray", vmin=0, vmax=255)
            axes[i, 1].set_title("GT mask", fontsize=9)
        else:
            axes[i, 1].imshow(
                np.zeros(img_arr.shape[:2]),
                cmap="gray",
                vmin=0,
                vmax=1,
            )
            axes[i, 1].set_title("GT mask (none - normal)", fontsize=9)

        axes[i, 1].axis("off")

        # Column 2: Overlay
        overlay = img_arr.copy()

        if mask_arr is not None:
            mask_resized = np.array(
                Image.fromarray(mask_arr).resize(
                    (img_arr.shape[1], img_arr.shape[0]),
                    Image.NEAREST,
                )
            )

            tinted = overlay.copy()
            tinted[mask_resized > 127] = [255, 0, 0]

            overlay = (0.6 * overlay + 0.4 * tinted).astype(np.uint8)

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay", fontsize=9)
        axes[i, 2].axis("off")

        # Column 3: Prediction information
        axes[i, 3].axis("off")

        verdict_var = row.get("verdict_variance", float("nan"))
        sem_ent = row.get("semantic_entropy", float("nan"))

        pred_text = (
            f"score={row['anomaly_score']:.2f}\n"
            f"verdict_var={verdict_var:.2f}\n"
            f"sem_ent={sem_ent:.2f}\n\n"
            f"pred_defect:\n{row['predicted_defect']}\n\n"
            f"pred_conf:\n{row['predicted_confidence']}\n\n"
            f"Observation:\n"
            f"{wrap_caption(row['example_observation'])}"
        )

        axes[i, 3].text(
            0,
            1,
            pred_text,
            va="top",
            ha="left",
            fontsize=8,
            family="monospace",
        )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved to {save_path}")

    plt.show()

##### Carpet

In [ ]:
display_predictions(
    full_df,
    category="carpet",
    filter_query="prediction_type == 'TP'"
)

In [ ]:
display_predictions(
    full_df,
    category="carpet",
    filter_query="prediction_type == 'TN'",
    n=3
)


In [ ]:
display_predictions(
    full_df,
    category="carpet",
    filter_query="prediction_type == 'FN'",
    n=3
)


##### Pill

In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'TP'",
    n=3
)


In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'TN'",
    n=3
)


In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'FN'",
    n=3
)


In [ ]:
display_predictions(
    full_df,
    category="pill",
    filter_query="prediction_type == 'FP'",
    n=3
)
